## Assignment for week 3.

Building a synthetic data generator through open source models.

The idea is to use gradio as a frontend: pick a model and a number of rows,
and get back a generated pandas dataframe.

In [1]:
# Loading dependencies
from dotenv import load_dotenv
import ast
import os

import torch
import gradio as gr
import pandas as pd
from transformers import pipeline

In [2]:
# Loading env
load_dotenv()
hf_key = os.getenv("HUGGING_FACE")

In [7]:
# Writing the function first

MODELS = [
    'meta-llama/Llama-3.2-3B-Instruct',
    'meta-llama/Meta-Llama-3.1-8B-Instruct',
]

_pipelines = {}


def get_pipeline(model: str):
    """Load (and cache) the text-generation pipeline for a model."""
    if model not in _pipelines:
        _pipelines[model] = pipeline(
            "text-generation",
            model=model,
            dtype='auto',
            token=hf_key,
            device_map='auto',
        )
    return _pipelines[model]


def generate_synthetic_data(model: str, length: int) -> pd.DataFrame:
    length = int(length)

    # Defining the prompt
    SYSTEM_PROMPT = (
        "You create synthetic data for the user, returned as a pandas dataframe. "
        "Respond with ONLY a Python dictionary - no explanation, no markdown fences. "
        "E.g. {'Fruit': ['apple', 'banana', 'pear'], 'calories': [100, 150, 20]}. "
        "Create at least 10 columns and exactly the requested number of rows per column. "
        "Use statistical distributions (normal, uniform, ...) for numeric values "
        "and nominal variables where suitable. Note that you do not need to use fruits, it is an example."
    )
    USER_PROMPT = (
        f"The number of rows requested is {length}. "
        "Create a Python dictionary that I can convert into a dataframe."
    )

    pipe = get_pipeline(model)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT},
    ]

    output = pipe(messages, max_new_tokens=2048)
    generated = output[0]['generated_text']
    # Chat models return the full message list; take the last (assistant) message
    text = generated[-1]['content'] if isinstance(generated, list) else generated

    # Extract the dictionary from the raw text (models sometimes add prose or fences)
    start = text.find('{')
    end = text.rfind('}')
    if start == -1 or end == -1 or end < start:
        raise ValueError(f"Model did not return a dictionary:\n{text}")
    data = ast.literal_eval(text[start:end + 1])

    return pd.DataFrame(data)

In [8]:
# Test the generator
df = generate_synthetic_data(MODELS[0], 10)
df

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

,Product,Category,Brand,Model,Year,Price,Rating,Quantity,Weight,Size,Color
0,A1,Electronics,BrandA,Model1,2020,1000.0,4.5,10,0.5,S,Red
1,A2,Electronics,BrandB,Model2,2021,1200.0,4.2,20,1.0,M,Blue
2,A3,Electronics,BrandA,Model3,2022,800.0,4.8,15,0.8,L,Green
3,A4,Electronics,BrandC,Model4,2023,900.0,4.1,12,0.7,XL,Yellow
4,A5,Electronics,BrandB,Model5,2024,1100.0,4.7,18,1.2,M,Red
5,A6,Electronics,BrandA,Model6,2025,1300.0,4.3,25,1.5,L,Blue
6,A7,Electronics,BrandC,Model7,2026,600.0,4.9,8,0.6,S,Green
7,A8,Electronics,BrandB,Model8,2027,700.0,4.0,22,1.1,XL,Yellow
8,A9,Electronics,BrandA,Model9,2028,800.0,4.6,11,0.9,M,Red
9,A10,Electronics,BrandC,Model10,2029,900.0,4.4,16,1.0,L,Blue


## Gradio frontend

In [9]:
# Gradio frontend: pick a model and a number of rows, get back a dataframe
test = gr.Interface(
    fn=generate_synthetic_data,
    inputs=[
        gr.Dropdown(choices=MODELS, value=MODELS[0], label="Model"),
        gr.Slider(minimum=1, maximum=1000, value=10, step=1, label="Number of rows"),
    ],
    outputs=gr.Dataframe(label="Synthetic data"),
    title="Synthetic data generator",
    description="Pick an open-source model and a number of rows; get back a generated dataframe.",
)

In [10]:
test.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
